# Wetterich equation ϕ4 flow in d = 3

Using the LPA approximation we solve the Wetterich equation for the $O(N=4)$ $\phi^4$ thery in $d=3$ dimensions.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scienceplots
plt.style.use(["science", "grid"])
import pandas as pd
from pathlib import Path
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import h5py

In [ ]:
RESULTS_DIR = "../results"

### Litim Regulator

In [ ]:
y = np.linspace(0,5,1000)

plt.figure(figsize=(8,6))
plt.plot(y, np.where(y < 1, y*(1/y - 1), 0))
plt.xlabel("y = p²/k²")
plt.ylabel("R_k/K²")

## Classical ϕ4 potential and derivatives

In [ ]:
data = np.loadtxt(RESULTS_DIR + "/V_classical.txt", skiprows=1, delimiter=",")
data_1 = np.loadtxt(RESULTS_DIR + "/V_classical_prime.txt", skiprows=1, delimiter=",")
data_2 = np.loadtxt(RESULTS_DIR + "/V_classical_doubleprime.txt", skiprows=1, delimiter=",")

ρ = data[:,0]
V = data[:,1]

ρ2 = data_1[:,0]
dV = data_1[:,1]

ρ3 = data_2[:,0]
ddV = data_2[:,1]

plt.figure(figsize=(8,6))
plt.plot(ρ,V,label="V")
plt.xlabel(rf"$\rho$")
plt.ylabel("V")
plt.plot(ρ2,dV, label="dV")
plt.plot(ρ3,ddV, label="ddV")
plt.legend()
plt.grid(alpha=0.5)

### initial RHS of Flow equation

In [ ]:
RHS = np.loadtxt(RESULTS_DIR + "/RHS.txt", skiprows=1, delimiter=",")

ρ = RHS[:,0]
RHS = RHS[:,1]

plt.figure(figsize=(8,6))
plt.plot(ρ,RHS)
plt.xlabel(rf"$\rho$")
# plt.xscale("log")
# plt.yscale("log")
plt.ylabel(rf"RHS($\rho$) at $k/\Lambda$ = 1")
plt.grid(alpha=0.5)

## Flow equation solution

Read fron CSV file

In [ ]:
CSV_PATH = Path(RESULTS_DIR + "/flow_adaptive.csv")

In [ ]:
def read_block(path, block_name):
    rows = []
    header = None
    inside = False
    with open(path) as f:
        for line in f:
            if line.startswith(f"# block: {block_name}"):
                inside = True
                continue
            if inside and line.startswith("# block:"):
                break          # next block started
            if inside and line.startswith("#"):
                continue       # other comment
            if inside:
                if header is None:
                    header = line.strip().split(",")
                else:
                    rows.append([float(x) for x in line.strip().split(",")])

    data = np.array(rows)
    rho = data[:, 0]
    k_vals = np.array([float(h.split("=")[1]) for h in header[1:]])

    # Older CSV files round very small k values to 0.000000 in the header.
    # Keep only positive finite entries so log(k) stays defined.
    valid = np.isfinite(k_vals) & (k_vals > 0)
    if not np.all(valid):
        k_vals = k_vals[valid]
        data = data[:, np.r_[True, valid]]

    matrix = data[:, 1:].T    # shape: (n_snapshots, n_rho)
    return rho, k_vals, matrix


rho_grid, k_values, V_grid = read_block(CSV_PATH, "V")
_, _, RHS_grid = read_block(CSV_PATH, "RHS")
t_values = np.log(k_values)

## Flow of the effective potential

In [ ]:
fig = plt.figure(figsize=(16, 6))

ax1 = fig.add_subplot(121)
mesh = ax1.pcolormesh(rho_grid, t_values, V_grid, shading="auto", cmap="viridis")
ax1.set_xlabel(r"$\rho$")
ax1.set_ylabel(r"$t = \ln k$")
ax1.set_title(r"$V_k(\rho)$ (2D)")
fig.colorbar(mesh, ax=ax1, label="V")

ax2 = fig.add_subplot(122, projection="3d")
Rho, T = np.meshgrid(rho_grid, t_values)
surf = ax2.plot_surface(Rho, T, V_grid, cmap="viridis", edgecolor="none")
ax2.set_xlabel(r"$\rho$")
ax2.set_ylabel(r"$t = \ln k$")
ax2.set_zlabel("V")
ax2.set_title(r"$V_k(\rho)$ (3D)")
fig.colorbar(surf, ax=ax2, shrink=0.6, pad=0.1, label="V")

plt.tight_layout()
plt.savefig(RESULTS_DIR + "/eff_pot_flow_1.pdf", format="pdf")
plt.show()

## Flow of the effective potential relative to the initial condition

In [ ]:
V_ref  = V_grid[0]          # UV snapshot  (largest k)
V_diff = (V_grid - V_ref)

Rho, T = np.meshgrid(rho_grid, t_values)

fig = plt.figure(figsize=(16, 12))

ax1 = fig.add_subplot(221)
mesh1 = ax1.pcolormesh(rho_grid, t_values, V_diff, shading="auto", cmap="RdBu_r")
ax1.set_xlabel(r"$\rho$")
ax1.set_ylabel(r"$t = \ln k$")
ax1.set_title(r"$V_k(\rho) - V_{\Lambda}(\rho)$ (2D)")
fig.colorbar(mesh1, ax=ax1, label=r"$\Delta V$")

ax2 = fig.add_subplot(222, projection="3d")
surf1 = ax2.plot_surface(Rho, T, V_diff, cmap="RdBu_r", edgecolor="none")
ax2.set_xlabel(r"$\rho$")
ax2.set_ylabel(r"$t = \ln k$")
ax2.set_zlabel(r"$\Delta V$")
ax2.set_title(r"$V_k(\rho) - V_{\Lambda}(\rho)$ (3D)")
fig.colorbar(surf1, ax=ax2, shrink=0.6, pad=0.1, label=r"$\Delta V$")

ax3 = fig.add_subplot(223)
mesh2 = ax3.pcolormesh(rho_grid, t_values, RHS_grid, shading="auto", cmap="RdBu_r")
ax3.set_xlabel(r"$\rho$")
ax3.set_ylabel(r"$t = \ln k$")
ax3.set_title(r"$\partial_t V_k(\rho)$ (2D)")
fig.colorbar(mesh2, ax=ax3, label=r"$\partial_t V$")

ax4 = fig.add_subplot(224, projection="3d")
surf2 = ax4.plot_surface(Rho, T, RHS_grid, cmap="RdBu_r", edgecolor="none")
ax4.set_xlabel(r"$\rho$")
ax4.set_ylabel(r"$t = \ln k$")
ax4.set_zlabel(r"$\partial_t V$")
ax4.set_title(r"$\partial_t V_k(\rho)$ (3D)")
fig.colorbar(surf2, ax=ax4, shrink=0.6, pad=0.1, label=r"$\partial_t V$")

plt.tight_layout()
plt.savefig(RESULTS_DIR + "/eff_pot_flow_2.pdf", format="pdf")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))

# Fix y-limits across all frames
y_min, y_max = V_grid.min(), V_grid.max()
margin = 0.05 * (y_max - y_min)

def animate(frame):
    ax.clear()
    ax.plot(rho_grid, V_grid[frame], "b-", linewidth=2)
    ax.set_xlabel(r"$\rho$")
    ax.set_ylabel("V")
    ax.set_title(rf"$V_k(\rho)$  at  $k = {k_values[frame]:.4f}$,  $t = {t_values[frame]:.3f}$")
    ax.set_ylim(y_min - margin, y_max + margin)
    ax.grid(alpha=0.4)

anim = FuncAnimation(fig, animate, frames=len(t_values), interval=120, repeat=True)
plt.tight_layout()
HTML(anim.to_jshtml())

In [ ]:
# convert rho grid to phi grid
phi_grid = np.sqrt(2 * rho_grid)
phi_full = np.concatenate([-phi_grid[::-1], phi_grid])  # mirror to negative phi

fig, ax = plt.subplots(figsize=(6, 5))
y_min, y_max = V_grid.min(), V_grid.max()
margin = 0.05 * (y_max - y_min)

def animate(frame):
    ax.clear()
    V_full = np.concatenate([V_grid[frame][::-1], V_grid[frame]])  # mirror V too
    ax.plot(phi_full, V_full, "b-", linewidth=2)
    ax.set_xlabel(r"$\phi$")
    ax.set_ylabel("V")
    ax.set_title(rf"$V_k(\phi)$  at  $k = {k_values[frame]:.4f}$,  $t = {t_values[frame]:.3f}$")
    ax.set_ylim(y_min - margin, y_max + margin)
    ax.grid(alpha=0.4)

anim = FuncAnimation(fig, animate, frames=len(t_values), interval=120, repeat=True)
plt.tight_layout()
HTML(anim.to_jshtml())

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

# Pick 5 target points linearly spaced in k (UV -> IR)
k_targets = np.linspace(k_values.max(), k_values.min(), 5)
frame_indices = np.array([np.argmin(np.abs(k_values - kt)) for kt in k_targets])

colors = plt.cm.viridis(np.linspace(0, 1, len(frame_indices)))
phi_full = np.concatenate([-phi_grid[::-1], phi_grid])

for idx, color in zip(frame_indices, colors):
    V_full = np.concatenate([V_grid[idx][::-1], V_grid[idx]])
    ax.plot(
        phi_full, V_full, color=color, lw=2,
        label=rf"$k={k_values[idx]:.4f}$, $t={t_values[idx]:.2f}$"
    )

ax.set_xlabel(r"$\phi$")
ax.set_ylabel(r"$V_k(\phi)$")
ax.set_title("Flow of the effective potential")
ax.grid(alpha=0.4)
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(RESULTS_DIR + "/eff_pot_flow_3.pdf", format="pdf")
plt.show()

## Flowing minimum of potential

In [ ]:
# Find the rho at which V is minimised for each snapshot
rho0 = rho_grid[np.argmin(V_grid, axis=1)]

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(k_values, rho0, "o-", color="tab:orange")
ax.set_xlabel("k")
ax.set_ylabel(r"$\rho_0(k)$")
ax.set_title(r"Flow of minimum $\rho_0(k)$")
ax.set_xscale("log")
ax.set_yscale("log")
ax.grid(alpha=0.4)
plt.tight_layout()

### Adaptive step size for RK4 time stepper

In [ ]:
timesteps_data = np.loadtxt(RESULTS_DIR + "/dt_values.txt", skiprows=2, delimiter=",")
timesteps = timesteps_data[:,1]
step_number = np.linspace(1,len(timesteps),len(timesteps))


fig = plt.figure(figsize=(9, 6))
plt.plot(step_number, abs(timesteps))
plt.yscale("log")
plt.grid(alpha=0.5)
plt.xlabel("step")
plt.title(rf"Step size $dt$")
plt.ylabel(rf"time step $|dt|$")
plt.show()